# 20.1 生存分析 / Survival Analysis (Kaplan-Meier & Cox PH)

**中文**:Part 20 是各类**专门主题**的合集。先讲**生存分析**——研究"**某事件多久后发生**"的方法:病人存活多久、用户多久流失、机器多久故障、贷款多久违约。它看似是回归(预测时间),却有一个让普通回归**彻底失效**的特性:**删失(censoring)**。研究结束时,很多对象**还没发生事件**(病人还活着、用户还没流失)——你只知道"至少活过了 T",不知道确切时间。**扔掉这些删失数据会严重低估生存**,硬套回归也会偏。生存分析专门优雅地处理删失。
**English**: Part 20 is a collection of **specialized topics**. First, **survival analysis** — methods for "**how long until an event happens**": how long a patient survives, when a user churns, when a machine fails, when a loan defaults. It looks like regression (predict a time) but has a feature that makes ordinary regression **completely fail**: **censoring**. At the study's end, many subjects **haven't had the event yet** (the patient is still alive, the user hasn't churned) — you only know "survived at least T," not the exact time. **Dropping censored data severely underestimates survival**, and forcing a regression is biased. Survival analysis handles censoring elegantly.

---

**中文**:两个核心工具:
**English**: Two core tools:

**中文**:
**① Kaplan-Meier(KM)生存曲线**:非参数地估计**生存函数 $S(t)=P(T>t)$**(活过时间 $t$ 的概率)。妙处在于它**正确利用删失**:每个事件发生的时刻,只用"当时还在观察中(at risk)"的人来算风险:
**English**:
**① Kaplan-Meier (KM) survival curve**: non-parametrically estimates the **survival function $S(t)=P(T>t)$** (probability of surviving past time $t$). Its beauty is **correctly using censored data**: at each event time, compute the risk using only those "still under observation (at risk)":

$$\hat S(t)=\prod_{t_i\le t}\Big(1-\frac{d_i}{n_i}\Big)$$

**中文**:$d_i$ 是时刻 $t_i$ 发生事件的数目,$n_i$ 是那一刻**仍在风险集里**的人数(还没发生事件、也还没被删失)。删失的对象在被删失前一直贡献于风险集——**信息没被浪费**。

**English**: $d_i$ is the number of events at time $t_i$, $n_i$ the number **still at risk** at that instant (no event yet and not yet censored). Censored subjects contribute to the risk set until censored — **no information wasted**.

**中文**:
**② Cox 比例风险模型(Cox PH)**:生存分析的"回归"——研究协变量(年龄、剂量、组别)如何影响风险。它建模**风险率(hazard)**:
**English**:
**② Cox Proportional Hazards (Cox PH)**: survival analysis's "regression" — how covariates (age, dose, group) affect risk. It models the **hazard rate**:

$$h(t\mid \mathbf x)=h_0(t)\,\exp(\boldsymbol\beta^\top\mathbf x)$$

**中文**:$h_0(t)$ 是**基线风险**(不用假设它的形状,所以叫"半参数"),$\exp(\beta_j)$ 是**风险比(hazard ratio, HR)**——协变量增加一个单位,风险变为原来的 $\exp(\beta_j)$ 倍。HR<1 = 保护因素(降低风险),HR>1 = 危险因素。用**偏似然(partial likelihood)** 估计 $\beta$(巧妙地把 $h_0(t)$ 消掉了)。
**English**: $h_0(t)$ is the **baseline hazard** (its shape needn't be assumed — hence "semi-parametric"), and $\exp(\beta_j)$ is the **hazard ratio (HR)** — a one-unit increase in the covariate multiplies the hazard by $\exp(\beta_j)$. HR<1 = protective (lowers risk), HR>1 = harmful. Estimate $\beta$ by **partial likelihood** (which cleverly cancels $h_0(t)$).

> 💡 **面试速查 / Interview cheat-sheet（★★ 医疗/风控/留存必考）**
> **中文**:生存分析=建模"事件发生时间", 核心难点是**删失(censoring)**——研究结束时事件还没发生, 只知"至少活过T"。**别扔删失数据**(严重低估)、别硬套回归。**Kaplan-Meier**:非参数生存曲线, 每个事件时刻只用风险集计算, 正确利用删失。**Log-rank 检验**:比较两组生存曲线是否有差异。**Cox PH**:半参数回归 h(t|x)=h₀(t)exp(βx), **风险比 HR=exp(β)**(<1保护/>1危险), 偏似然估计(消掉基线风险), 关键假设=**比例风险**(HR 不随时间变, 可用 Schoenfeld 残差检验)。用途:客户流失/LTV、医疗生存、设备可靠性、贷款违约。区别 churn 分类:生存分析用上"还没流失"的人 + 预测"何时"而非"是否"。
> **English**: Survival analysis = model "time to event"; the core difficulty is **censoring** — at study end the event hasn't happened, you only know "survived at least T." **Don't drop censored data** (severe underestimation) or force a regression. **Kaplan-Meier**: a non-parametric survival curve computed from the risk set at each event time, correctly using censoring. **Log-rank test**: compare two groups' survival curves. **Cox PH**: semi-parametric regression h(t|x)=h₀(t)exp(βx), **hazard ratio HR=exp(β)** (<1 protective / >1 harmful), estimated by partial likelihood (cancels the baseline hazard), key assumption = **proportional hazards** (HR constant over time; check with Schoenfeld residuals). Uses: churn/LTV, medical survival, equipment reliability, loan default. Vs churn classification: survival uses "not-yet-churned" subjects + predicts "when" not just "whether."


In [ ]:

# ============================================================
# 合成生存数据(含删失)/ synthetic survival data with censoring
# 中文:两组(对照/处理), 处理把风险减半(真实HR=0.5)。指数分布生成事件时间, 另有删失时间。
#      观测到的 = min(事件时间, 删失时间); observed=1 表示真事件, 0 表示删失(研究结束还没发生)。
# English: two groups (control/treatment); treatment halves the hazard (true HR=0.5). Exponential event
#      times, plus censoring times. Observed = min(event, censor); observed=1 = real event, 0 = censored.
# ============================================================
import numpy as np, matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy import stats
rng=np.random.default_rng(0)
N=500; true_HR=0.5
group=rng.integers(0,2,N)                                    # 0=对照, 1=处理 / control, treatment
hazard=0.1*np.where(group==1, true_HR, 1.0)                  # 处理组风险减半 / treatment halves hazard
T_event=rng.exponential(1/hazard)                           # 真实事件时间 / true event time
T_censor=rng.exponential(1/0.04, N)                         # 删失时间 / censoring time
time=np.minimum(T_event, T_censor)                          # 实际观测时间 / observed time
observed=(T_event<=T_censor).astype(int)                    # 1=事件, 0=删失 / event vs censored
print(f"N={N}, 观测到事件 {observed.sum()}, 删失 {(1-observed).sum()} ({(1-observed).mean():.0%})")
print("删失=研究结束时还没发生事件的对象, 不能扔掉 / censored = no event by study end, don't drop them")


**中文**:先从零实现 **Kaplan-Meier** 曲线,对两组分别估计生存函数并可视化。KM 曲线是阶梯状的——每次有事件发生就下降一格,删失点用小竖线标记(它们贡献了风险集但不导致下降)。
**English**: First implement **Kaplan-Meier** from scratch, estimating the survival function for each group and visualizing. KM curves are step functions — dropping at each event, with censoring points marked by ticks (they contribute to the risk set but cause no drop).


In [ ]:

# ============================================================
# 从零实现 Kaplan-Meier / Kaplan-Meier from scratch
# ============================================================
def kaplan_meier(time, observed):
    event_times=np.sort(np.unique(time[observed==1]))       # 所有发生事件的时刻 / distinct event times
    S=1.0; ts=[0.0]; Ss=[1.0]
    for t in event_times:
        n_at_risk=(time>=t).sum()                           # 该时刻仍在风险中的人数 / at risk
        d=((time==t)&(observed==1)).sum()                   # 该时刻的事件数 / events at t
        S*=(1 - d/n_at_risk)                                # 乘上存活比例 / multiply survival fraction
        ts.append(t); Ss.append(S)
    return np.array(ts), np.array(Ss)

t_ctrl,S_ctrl=kaplan_meier(time[group==0], observed[group==0])
t_trt, S_trt =kaplan_meier(time[group==1], observed[group==1])
def median_surv(ts,Ss):                                     # 中位生存时间(S 首次≤0.5)/ median survival
    idx=np.where(Ss<=0.5)[0]; return ts[idx[0]] if len(idx) else np.inf
print(f"对照组中位生存时间 / control median survival: {median_surv(t_ctrl,S_ctrl):.1f}")
print(f"处理组中位生存时间 / treatment median survival: {median_surv(t_trt,S_trt):.1f}  (更长=处理有效)")


**中文**:两组的 KM 曲线肉眼看有差异,但**这个差异是真实的还是随机波动**？用 **log-rank 检验**回答:它在每个事件时刻比较"每组的实际事件数"与"若两组无差异时的期望事件数",汇总成一个卡方统计量。
**English**: The two KM curves look different by eye, but **is the difference real or random noise**? The **log-rank test** answers: at each event time it compares "each group's observed events" with "expected events if the groups were identical," aggregating into a chi-square statistic.


In [ ]:

# ============================================================
# 从零实现 log-rank 检验 / log-rank test from scratch
# ============================================================
def logrank_test(time, observed, group):
    event_times=np.sort(np.unique(time[observed==1]))
    O1=E1=V=0.0                                             # 组1的观测事件 / 期望 / 方差
    for t in event_times:
        at_risk=time>=t; n=at_risk.sum(); n1=(at_risk&(group==1)).sum()
        d=((time==t)&(observed==1)).sum(); d1=((time==t)&(observed==1)&(group==1)).sum()
        O1+=d1; E1+=d*n1/n                                  # 期望事件数(按人数比例分配)/ expected
        if n>1: V+=d*(n1/n)*(1-n1/n)*(n-d)/(n-1)            # 超几何方差 / hypergeometric variance
    chi2=(O1-E1)**2/V; p=1-stats.chi2.cdf(chi2,df=1)
    return chi2, p
chi2,p=logrank_test(time,observed,group)
print(f"Log-rank 检验: χ²={chi2:.1f}, p={p:.2e}")
print("→ p 极小 → 两组生存曲线显著不同 → 处理确实影响了生存 / groups differ significantly")


**中文**:最后从零实现 **Cox 比例风险模型**,用**偏似然**估计风险比。在每个事件时刻,"这个事件恰好发生在观测个体身上"的条件概率里,基线风险 $h_0(t)$ 被约掉了——所以不用假设它的形状。看它能否还原真实 HR=0.5。
**English**: Finally implement **Cox Proportional Hazards** from scratch, estimating the hazard ratio via **partial likelihood**. At each event time, in the conditional probability "this event happened to the observed individual," the baseline hazard $h_0(t)$ cancels — so its shape needn't be assumed. See if it recovers the true HR=0.5.


In [ ]:

# ============================================================
# 从零实现 Cox PH(偏似然)/ Cox PH via partial likelihood
# ============================================================
X=group.reshape(-1,1).astype(float)                        # 协变量:组别 / covariate: group
def neg_log_partial_likelihood(beta):
    ll=0.0
    for i in range(N):
        if observed[i]==0: continue                        # 只有事件贡献偏似然 / only events contribute
        risk_set = time>=time[i]                           # 风险集:此刻还没事件的人 / at-risk set
        ll += X[i]@beta - np.log(np.sum(np.exp(X[risk_set]@beta)))  # log(个体风险/风险集总风险)
    return -ll
res=minimize(neg_log_partial_likelihood, [0.0], method="BFGS")
beta_hat=res.x[0]; HR=np.exp(beta_hat)
# 用逆 Hessian 估标准误 / SE from inverse Hessian
se=np.sqrt(res.hess_inv[0,0]); ci=(np.exp(beta_hat-1.96*se), np.exp(beta_hat+1.96*se))
print(f"Cox β={beta_hat:.3f}, 风险比 HR=exp(β)={HR:.3f} (真值 {true_HR})")
print(f"HR 95% 置信区间 / CI: [{ci[0]:.3f}, {ci[1]:.3f}]  (不含1 → 显著)")
print(f"解读:处理组风险是对照组的 {HR:.2f} 倍 = 降低 {(1-HR)*100:.0f}% 风险 / treatment cuts hazard ~{(1-HR)*100:.0f}%")


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,2,figsize=(14,5))
# ① KM 生存曲线 / KM survival curves
ax[0].step(t_ctrl,S_ctrl,where="post",color="#C44E52",lw=2,label="对照组 control")
ax[0].step(t_trt,S_trt,where="post",color="#4C72B0",lw=2,label="处理组 treatment")
# 删失点标记 / censoring marks
for g,c in [(0,"#C44E52"),(1,"#4C72B0")]:
    ct=time[(group==g)&(observed==0)]
    tt,ss=(t_ctrl,S_ctrl) if g==0 else (t_trt,S_trt)
    for x in ct[ct<tt.max()]:
        yy=ss[np.searchsorted(tt,x,side="right")-1]; ax[0].plot(x,yy,"|",color=c,ms=8,alpha=0.5)
ax[0].axhline(0.5,ls=":",color="gray",label="中位生存")
ax[0].set_title("Kaplan-Meier 生存曲线(竖线=删失)/ KM curves (ticks=censored)"); ax[0].set_xlabel("时间 t"); ax[0].set_ylabel("生存概率 S(t)"); ax[0].legend(fontsize=9); ax[0].set_ylim(0,1)
# ② 若错误地扔掉删失 vs 正确处理 / dropping censored (wrong) vs correct
def naive_drop(time,observed):  # 错误做法:只用观测到事件的, 当作精确时间 / WRONG: drop censored
    et=np.sort(time[observed==1]); return et, 1-np.arange(1,len(et)+1)/len(et)
tn,Sn=naive_drop(time[group==0],observed[group==0])
ax[1].step(t_ctrl,S_ctrl,where="post",color="#55A868",lw=2,label="KM(正确处理删失)")
ax[1].step(tn,Sn,where="post",color="#C44E52",lw=2,ls="--",label="扔掉删失(错误→低估生存)")
ax[1].set_title("扔删失=低估生存 / dropping censored underestimates survival"); ax[1].set_xlabel("时间 t"); ax[1].set_ylabel("S(t)"); ax[1].legend(fontsize=9); ax[1].set_ylim(0,1)
plt.tight_layout(); plt.savefig("/tmp/adv01_viz.png",dpi=80); plt.show()
print("右图:错误地扔掉删失数据, 生存曲线被系统性拉低——删失处理是生存分析的灵魂")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **删失是生存分析的灵魂**:研究结束时还没发生事件的对象(病人还活着、用户还没流失),不是"缺失数据"——它们携带着"至少活过 T"的宝贵信息。KM 正确地把它们纳入风险集;而右图显示,**若把删失当成"没发生就扔掉",生存曲线会被系统性地拉低**(严重悲观)。这就是为什么不能用普通回归/分类硬套时间数据。
2. **KM + log-rank + Cox 是黄金三件套**:KM 画出生存曲线(描述)、log-rank 检验组间差异(推断)、Cox PH 量化协变量的风险比(回归)。本例 Cox 精确还原了真实 HR=0.5(处理降低 48% 风险),log-rank p 值极小确认差异显著。**风险比 HR 是最常用的汇报量**——"HR=0.5"直观表示"风险减半",比"系数 β=−0.65"好懂得多。
3. **诚实的关键假设:比例风险**:Cox 假设**风险比不随时间变化**(两组的风险曲线始终成固定倍数)。如果处理的效果随时间衰减(如药效前期强后期弱),这个假设就被违反,HR 会误导。**必须检验**(Schoenfeld 残差、看 KM 曲线是否交叉)。此外,Cox 的 HR 是"平均"效应,同样可能掩盖异质性(和 Part 19 一个道理)。

**English**:
1. **Censoring is the soul of survival analysis**: subjects without the event at study end (patient alive, user not churned) are not "missing data" — they carry the valuable information "survived at least T." KM correctly folds them into the risk set; the right plot shows that **treating censored as "didn't happen, drop it" systematically pulls the survival curve down** (severely pessimistic). This is why you can't force ordinary regression/classification onto time data.
2. **KM + log-rank + Cox is the golden trio**: KM draws the survival curve (description), the log-rank test compares groups (inference), and Cox PH quantifies covariates' hazard ratio (regression). Here Cox exactly recovers the true HR=0.5 (treatment cuts risk 48%), and the log-rank p-value confirms significance. **The hazard ratio HR is the most-reported quantity** — "HR=0.5" intuitively means "risk halved," far clearer than "coefficient β=−0.65."
3. **The key honest assumption: proportional hazards**: Cox assumes the **hazard ratio is constant over time** (the two groups' hazards always in a fixed ratio). If the treatment's effect decays over time (a drug strong early, weak later), this is violated and the HR misleads. **You must test it** (Schoenfeld residuals; check whether KM curves cross). Also, Cox's HR is an "average" effect and can hide heterogeneity (same lesson as Part 19).

> 💼 **实战视角 / Practical angle**
> **中文**:生存分析在**互联网**里极其实用, 但常被忽视:①**客户流失/留存**——别用"30天是否流失"的分类(丢掉了还没流失的人 + 只知是否不知何时), 用生存分析能算"预期留存时长"、画留存曲线、算 LTV;②**订阅/续费预测**;③**设备故障、贷款违约、A/B 里"到达某目标的时间"**。工具:`lifelines`(Python, KaplanMeierFitter/CoxPHFitter)、`scikit-survival`。落地要点:①想清楚"事件"和"删失"怎么定义(数据截断日 = 删失);②**检验比例风险假设**(违反就用时变系数或分层 Cox);③离散时间生存可用逻辑回归+人-时展开;④深度学习版:DeepSurv、DeepHit。面试金句:*"生存分析处理删失(还没发生事件的人不能扔); KM 非参数画生存曲线、log-rank 比组、Cox PH 半参数回归给风险比 HR=exp(β); 关键假设是比例风险, 用途包括留存/LTV/违约/故障——比'X天是否流失'的分类信息量大得多。"*
> **English**: Survival analysis is highly useful yet often overlooked in tech: ① **churn/retention** — don't use "churned within 30 days" classification (it discards not-yet-churned users and knows only whether, not when); survival analysis gives "expected retention time," retention curves, and LTV; ② **subscription/renewal prediction**; ③ **equipment failure, loan default, "time to reach a goal" in A/B**. Tools: `lifelines` (Python, KaplanMeierFitter/CoxPHFitter), `scikit-survival`. Deployment keys: ① clearly define "event" and "censoring" (data cutoff date = censoring); ② **test proportional hazards** (violated → time-varying coefficients or stratified Cox); ③ discrete-time survival via logistic regression + person-period expansion; ④ deep-learning versions: DeepSurv, DeepHit. Interview line: *"Survival analysis handles censoring (don't drop not-yet-event subjects); KM draws survival curves non-parametrically, log-rank compares groups, Cox PH is semi-parametric regression giving the hazard ratio HR=exp(β); the key assumption is proportional hazards; uses include retention/LTV/default/failure — far more informative than 'churned within X days' classification."*

---
### 小结 / Summary
- **中文**:生存分析建模事件发生时间, 核心是正确处理删失(还没发生事件的对象); 别扔、别硬套回归。
- **English**: Survival analysis models time-to-event; the core is correctly handling censoring (not-yet-event subjects); don't drop them or force a regression.
- **中文**:KM(非参数生存曲线)+ log-rank(比组)+ Cox PH(HR=exp(β), 偏似然, 消基线风险)=黄金三件套。
- **English**: KM (non-parametric survival curve) + log-rank (compare groups) + Cox PH (HR=exp(β), partial likelihood, cancels baseline hazard) = the golden trio.
- **中文**:关键假设=比例风险(需检验); 用途:流失/LTV/违约/故障——比"是否流失"的分类强得多。
- **English**: Key assumption = proportional hazards (must test); uses: churn/LTV/default/failure — far stronger than "whether churned" classification.
